# Processamento de Linguagem Natural
## Exercícios TC.10 e PC.10

**Disciplina:** Processamento de Linguagem Natural
**Professor:** Fabrício Galende Marques de Carvalho
**Aluno(a):** _(preencher)_
**Data:** 24 de agosto de 2026

Este notebook contém o código-fonte executável referente à demonstração do exercício **TC.10** e à resolução completa do exercício **PC.10**. A discussão conceitual detalhada de cada exercício está disponível na apresentação em PDF que acompanha este material.

### Procedimento de execução

1. Criar e ativar um ambiente virtual Python 3 (recomendado):

   ```bash
   python3 -m venv venv
   source venv/bin/activate        # Linux/macOS
   venv\Scripts\activate           # Windows
   ```

2. Instalar as dependências listadas em `requirements.txt` (o arquivo já inclui o modelo `pt_core_news_sm` do spaCy):

   ```bash
   pip install -r requirements.txt
   ```

3. Abrir este notebook com o Jupyter:

   ```bash
   jupyter notebook TC10_PC10_PLN.ipynb
   ```

4. Executar as células em sequência (`Run All`). A função `assistente_virtual()` (seção PC.10) inicia um loop interativo via `input()` e está, por isso, comentada na célula de definição — para testá-la manualmente, descomente a chamada e execute a célula individualmente, pois o `input()` bloquearia a execução automática de "Run All".

## TC.10 — Emojis e símbolos não formais em sistemas de PLN

**Enunciado:** Por que a utilização de emojis ou outros símbolos não presentes na linguagem textual formal podem dificultar a operação de um sistema de PLN? Cite um exemplo prático de sentença textual comumente presente em diálogos que faz uso de emojis e ilustre qual a dificuldade pode ser gerada ao sistema de PLN.

Este exercício é essencialmente teórico e não exige, a rigor, o desenvolvimento de um programa. Ainda assim, a célula de código a seguir demonstra na prática, com uma frase real contendo emojis, o problema discutido em detalhes na apresentação em PDF: os modelos de PLN são treinados sobre vocabulário linguístico formal e não possuem representação adequada para símbolos pictográficos.

In [14]:
import spacy
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

nlp = spacy.load("pt_core_news_sm")

In [15]:
frase = "Adorei o produto \U0001F60D\U0001F44F, mas a entrega atrasou muito \U0001F621"

print("Frase de entrada:")
print(frase)
print()

doc = nlp(frase)
print(f"{'Token':<10}{'POS':<8}{'Lema':<10}")
print("-" * 28)
for token in doc:
    print(f"{token.text:<10}{token.pos_:<8}{token.lemma_:<10}")

print()
print("Tokenização com NLTK (word_tokenize):")
print(word_tokenize(frase, language="portuguese"))

Frase de entrada:
Adorei o produto 😍👏, mas a entrega atrasou muito 😡

Token     POS     Lema      
----------------------------
Adorei    VERB    Adorei    
o         DET     o         
produto   NOUN    produto   
😍         ADJ     😍         
👏         NOUN    👏         
,         PUNCT   ,         
mas       CCONJ   mas       
a         DET     o         
entrega   NOUN    entrega   
atrasou   VERB    atrasar   
muito     ADV     muito     
😡         VERB    😡         

Tokenização com NLTK (word_tokenize):
['Adorei', 'o', 'produto', '😍👏', ',', 'mas', 'a', 'entrega', 'atrasou', 'muito', '😡']


### Discussão do resultado

Observando a saída acima, é possível notar dois problemas concretos:

1. **Classificação gramatical (POS) incoerente:** o spaCy atribui aos emojis 😍, 👏 e 😡 as classes `ADJ`, `NOUN` e `VERB`, respectivamente — rótulos sem qualquer sentido linguístico real, pois o modelo estatístico nunca observou esses símbolos durante o treinamento e faz uma estimativa forçada a partir do contexto. O lema atribuído a cada emoji é o próprio símbolo, ou seja, nenhuma informação semântica é efetivamente extraída.
2. **Divergência de tokenização:** o tokenizador do spaCy separa 😍 e 👏 em dois tokens distintos, enquanto o `word_tokenize` do NLTK mantém os dois símbolos agrupados em um único token (`'😍👏'`). Isso mostra que ferramentas diferentes de PLN não possuem um tratamento padronizado para esse tipo de símbolo, o que compromete a reprodutibilidade e a comparabilidade dos resultados entre pipelines distintas.



## PC.10 — Assistente virtual baseado em tags POS

**Enunciado:** Fazendo uso de tags do tipo POS, desenvolva um assistente virtual simples que identifique se o usuário está fazendo uma pergunta ou se é afirmação ou comando. *Hint: olhe para a primeira palavra da frase e seu POS.*



In [16]:
PALAVRAS_INTERROGATIVAS = {
    "que", "quem", "qual", "quais", "quando", "onde",
    "como", "quanto", "quantos", "quanta", "quantas",
}


def identificar_tipo_frase(frase: str) -> str:
    """Classifica uma frase em 'Pergunta', 'Comando' ou 'Afirmação'
    a partir da classe gramatical (POS) da primeira palavra."""
    frase = frase.strip()
    if not frase:
        return "Afirmação"

    frase_normalizada = frase[0].lower() + frase[1:]

    doc = nlp(frase_normalizada)
    primeiro = doc[0]
    pos = primeiro.pos_
    modo = primeiro.morph.get("Mood")

    if pos in ("PRON", "DET", "ADV") and primeiro.text.lower() in PALAVRAS_INTERROGATIVAS:
        return "Pergunta"

    if pos in ("VERB", "AUX") and modo and modo[0] in ("Imp", "Sub"):
        return "Comando"

    if frase.endswith("?"):
        return "Pergunta"

    return "Afirmação"

In [17]:
frases_teste = [
    "Você fechou a porta?",
    "Feche a porta, por favor.",
    "O gato está dormindo no sofá.",
    "Quem chamou você ontem?",
    "Envie o relatório até amanhã.",
    "Ela chegou cedo para a reunião.",
    "Onde fica a estação mais próxima?",
    "Abra a porta.",
]

print(f"{'Frase':<38}{'Classificação'}")
print("-" * 55)
for frase in frases_teste:
    print(f"{frase:<38}{identificar_tipo_frase(frase)}")

Frase                                 Classificação
-------------------------------------------------------
Você fechou a porta?                  Pergunta
Feche a porta, por favor.             Comando
O gato está dormindo no sofá.         Afirmação
Quem chamou você ontem?               Pergunta
Envie o relatório até amanhã.         Comando
Ela chegou cedo para a reunião.       Afirmação
Onde fica a estação mais próxima?     Pergunta
Abra a porta.                         Comando


In [18]:
def assistente_virtual():
    """Loop interativo simples que classifica cada frase digitada pelo usuário.
    Digite 'sair' para encerrar."""
    print("Assistente virtual iniciado. Digite 'sair' para encerrar.\n")
    while True:
        frase = input("Você: ")
        if frase.strip().lower() == "sair":
            print("Assistente: Até logo!")
            break
        tipo = identificar_tipo_frase(frase)
        print(f"Assistente: identifiquei isso como uma {tipo.lower()}.\n")




In [19]:
assistente_virtual()

Assistente virtual iniciado. Digite 'sair' para encerrar.

Assistente: Até logo!
